# Spår B – Klassificering (identifiera “högpris-områden”) 

Bakgrundshistoria
 Kunden vill bygga ett verktyg som kan användas när ett nytt område (eller ett område som inte är analyserat ännu) dyker upp. De har tillgång till områdets egenskaper (geografi, hushåll, inkomstnivåer osv.) men de vet inte bostadsvärdet än.
 För att prioritera var man ska lägga tid och resurser vill de kunna få en snabb klassning: ”ser detta ut som ett högpris-område?”

Uppdrag
- Skapa en binär target high_value i historisk data:
high_value = 1 om median_house_value ligger i topp 20%
annars 0
- Träna en modell som förutsäger high_value utifrån X-variablerna (t.ex. median_income, geografi, hushållsvariabler, ocean_proximity, osv.)
- När du tränar modellen får du bara använda X-variablerna (alla kolumner utom median_house_value och din target high_value)

Viktigt
 När modellen används på nya områden finns inte median_house_value tillgängligt.
 Därför får median_house_value inte ingå som input-feature i klassificeringen (den används endast för att skapa target i träningsdatan).

Syfte
Att kunna prioritera områden och resonera om vilka typer av fel som är mest problematiska (t.ex. missa ett högpris-område vs flagga ett område i onödan).
Målet är inte att gissa priset, utan att snabbt flagga områden som sannolikt är högpris så att teamet kan prioritera.


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
    cross_val_predict
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from scipy.sparse import hstack

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score
)

sns.set_context("notebook")



## Load data

In [ ]:
df = pd.read_csv("housing.csv")

print("\n Dataset columns excerpt:")
print(df.head())

## 1) Dataförståelse & EDA 

 Krav:
- Visa datasetets storlek, datatyper och vilka features som finns.
- Kontrollera saknade värden och beskriv kort hur du hanterar dem.
- Minst 2 relevanta figurer/tabeller + kort tolkning.

In [ ]:
print("\nDatasetets storlek:\n", df.shape)
print("---------------------------")

In [ ]:
print("\nDatatyper:\n", df.dtypes)
print("---------------------------")

In [ ]:
print("\nFeatures i datasetet:\n", df.columns)
print("---------------------------")

### Summering av dataset

- Datsetet innehåller 20640 observationer och 10 variabler.
- Av dem 10 variablerna är 9 numeriska variabler av typerna float64 och 1 är en kategorisk variabel av typen objekt - *ocean_proximity*.
- Target är baserat på *median_house_value*. 


In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

scatter = plt.scatter(
    df["longitude"],
    df["latitude"],
    c=df["median_house_value"],
    cmap= plt.get_cmap("viridis"),
    alpha=0.7
)
legend1 = ax.legend(*scatter.legend_elements(),
                    loc="lower left", title="Bostadsvärde")
ax.add_artist(legend1)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Geografisk fördelning av bostadsvärden")
ax.grid(True)

plt.show()

#### Geografisk fördelning av bostadsvärden

Den geografiska scatterploten visar tydliga kluster där bostadsvärden är högre, framför allt längs kusten.
Detta bekräftar att geografi är en starkt förklarande variabel och motiverar varför longitude och latitude bör ingå i modellen.
Figuren hjälper även stakeholders att intuitivt förstå varför vissa områden systematiskt har högre värden.

In [ ]:
corr_matrix = df[["longitude","latitude","housing_median_age","total_rooms", "total_bedrooms","population", "households","median_income","median_house_value"]].corr(numeric_only=True)

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix,cmap="viridis", center=0)
plt.title("Korrelationsmatris (numeric features)")
plt.show()

#### Korrelationsmatris (numeric features)

Det finns ett tydligt samband mellan *median_income* och *median_house_value* - en nog ganska förväntad korrelation. Det är också väldigt starka korrelationer mellan variablerna *total_rooms*, *total_bedrooms*, *population* och *households* vilket tyder på att det beskriver liknande strukturella egenskaper i områdena.
Modellen kan därför behöva regularisering för att hantera dessa överlappande signaler.
*Longitude* och *latitude* har svagare korrelationer med bostadsvärde i matrisen, men deras effekt är icke‑linjär så syns bättre i scatterploten.

### Saknade värden

In [ ]:
print("\nSaknade värden (antal):\n", df.isna().sum())
print("-------------------------------")
print("\nSaknade värden (%):\n", df.isna().sum()/len(df)*100)
print("-------------------------------")

### Summering av hantering av saknade värden

Andelen saknade värden är väldigt låg - *total_bedrooms*(1%). För en numerisk variabel med så lite saknade värden är SimpleImputer(strategy="median") den mest effektiva lösningen och best practice. För att minska leakage kommer saknade värden imputeras i pipeline genom att median-imputering beräknas på träningsdatan och sedan att samma transformation appliceras på testdata. 

---


## 2) Target Engineering (Skapa y)

Steg 1 av uppdraget: 
- Skapa en binär target high_value i historisk data:
- high_value = 1 om median_house_value ligger i topp 20%
- annars 0

In [ ]:
threshold = df["median_house_value"].quantile(0.80) #Skapar binär target high_value -> använder quantile(0.80) för att hitta gränsen för topp 20% dyraste bostäder
df["high_value"] = (df["median_house_value"] >= threshold).astype(int) #Skapar en ny binär kolumn: high_value baserat på threshold.

y = df["high_value"]

### Summering av target engineering

För att omvandla problemet till en binär klassificeringsuppgift definierades en ny target‑variabel, *high_value*.
Tröskeln sattes till 80:e percentilen av , vilket motsvarar de 20% dyraste bostäderna.
Den definieras som 1 om *median_house_value* ligger i topp 20%, annars 0.
Detta gör det möjligt att identifiera premiumområden utan att behöva förutsäga exakta priser.

---

## 3) Feature Engineering (transformera X)

Variabeln income_cat skapades för att gruppera median_income i fem socioekonomiskt relevanta nivåer.
Den används för analys och kan även användas för stratifiering

In [ ]:
df["income_cat"] = pd.cut(
    df["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
) 

X = df.drop(["median_house_value", "high_value", "income_cat"], axis=1)

In [ ]:
print("X:", X.shape, "y:", y.shape)
print("---------------------------")

print("\n Income category distribution (whole dataset):")
print(df["income_cat"].value_counts())
print("---------------------------")

### Summering av feature engineering

Income cat skapar kategoriska intervall från den kontinuerliga variabeln. Den omvandlar en numerisk kolumn till 5 ordnade kategorier som sedan kan användas för stratifierat urval.
Gränserna för intervallen valdes utifrån tre principer: den statistiska fördelningen, socioekonomisk relevans samt att varje intervall skulle innehålla tillräckligt många observationer för att möjliggöra stratifiering.

---

## 4) Split + preprocessing

 Krav:
- Dela datan i train och test.
- Klassificering: använd stratifierad split (stratify) så att klasserna fördelas rimligt i train/test.
- Preprocessing ska göras på ett sätt som undviker att testdata påverkar träningen.

### Train/test split

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.25,
    random_state = 42,
    stratify= y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train postive rate:", y_train.mean().round(3), "Test postive rate:", y_test.mean().round(3))


display(X_train.head()) 

print("\nSaknade värden (train)")
print(X_train.isna().sum().sort_values(ascending=False).head(10))

print("\nSammanfattning (train)")
print(X_train.describe().T.head(10))

### Summering av train/test split

Vi delar upp datan i tränings- och testset med en 75/25-fördelning. 
Stratify=y_full garanterar att andelen misstänkta och icke-misstänkta fall är proportionerligt lika i både tränings- och testset.

### Preprocessing


In [ ]:
numeric_features = ["longitude","latitude","housing_median_age","total_rooms","total_bedrooms","population","households","median_income"]
categorical_features = ["ocean_proximity"]


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)



### Summering av preprocess

2 transformers skapas:
- numerisk transformer hanterar saknade värden med medianvärdet och standardiserar sedan värdena med StandardScaler.
- kategorisk transformer fyller saknade värden fyller saknade värden med "most_frequent",  följt av OneHotEncoder som omvandlar textvärden till binära kolumner som modellen kan tolka.

Dem två transformers slås ihop i en ColumnTransformer som applicerar rätt transformer på rätt kolumner.


---

## 5) Modellering

 Krav:
- Skapa en baseline.
- Träna minst två ytterligare modeller (totalt minst 3 inkl baseline).
- Jämför modellerna med en tydlig utvärderingsmetod (t.ex. cross-validation eller valideringsupplägg).

In [ ]:
lr = LogisticRegression(max_iter=500, random_state=42)
rf =  RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
gb = GradientBoostingClassifier(random_state=42)

pipe_lr = Pipeline([("preprocess", preprocess), ("model", lr)])
pipe_rf = Pipeline([("preprocess", preprocess), ("model", rf)])
pipe_gb = Pipeline([("preprocess", preprocess), ("model", gb)])


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = "f1"

In [ ]:
baseline_rows = []

for name, pipe in [("LogisticRegression", pipe_lr), ("RandomForest", pipe_rf), ("GradientBoosting", pipe_gb)]:
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=SCORING)
    baseline_rows.append({"model": name, "mean": scores.mean(), "std": scores.std()})

baseline_table = pd.DataFrame(baseline_rows).sort_values("mean", ascending=False)
baseline_table



In [ ]:
top2_names = baseline_table["model"].head(2).tolist()
print("Top-2 selected:", top2_names)

### Summering av modellering

Logistic Regression används som baseline modell då den är den enklaste linjöra modellen med få hyperparametrar som funkar som en referenspunkt för mer avancerade modeller.
I datasetet finns icke-linjära geografiska mönster, korrelerade variabler (*total_rooms*, *total_bedrooms*, *population* och *households*), obalanserad target samt en blandning av numeriska och kategoriska features.

Med tanke på komplexitetsnivån används två olika ensemble modeller: 
- Random Forest (Bagging-ensemble) och 
- Gradient Boosting (Boosting-ensemble).
För modelljämförelsen används F1‑score, eftersom det balanserar precision och recall och hanterar den obalanserade targetfördelningen på ett rättvist sätt.


---

## 6) Välj och optimera en modell

 Krav:
- Välj en modell baserat på din jämförelse.
- Optimera den valda modellen med hyperparameter-tuning (t.ex. GridSearchCV). Du väljer själv vilka parametrar som är relevanta
- Beskriv kort vad du optimerade och vilken metric du optimerade mot.

In [ ]:
param_grid_rf = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 15],
    "model__min_samples_leaf": [1, 5],
    "model__max_features": ["sqrt"]
}

grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_rf.fit(X_train, y_train)

print("Best RF params:", grid_rf.best_params_)
print("Best RF CV score:", round(grid_rf.best_score_, 3))

In [ ]:
res_rf = pd.DataFrame(grid_rf.cv_results_)
res_rf.sort_values("rank_test_score").head(8)

In [ ]:
param_grid_gb = {
    "model__n_estimators": [200, 400],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_depth": [2, 3],
    "model__subsample": [0.7, 1.0],
    "model__min_samples_leaf": [1, 10]
}

grid_gb = GridSearchCV(
    estimator=pipe_gb,
    param_grid=param_grid_gb,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_gb.fit(X_train, y_train)

print("Best GB params:", grid_gb.best_params_)
print("Best GB CV score:", round(grid_gb.best_score_, 3))

In [ ]:
res_gb = pd.DataFrame(grid_gb.cv_results_)
res_gb.sort_values("rank_test_score").head(8)

In [ ]:
print("GB best CV f1:", round(grid_gb.best_score_, 3))
print("RF best CV f1:", round(grid_rf.best_score_, 3))

winner_name = "GradientBoosting"
winner = grid_gb.best_estimator_

print("Winner:", winner_name)

### Summering av modell optimering





---

## 7) Slutlig utvärdering på testdata + rekommendation

 Krav:
- Utvärdera din slutliga modell på testdata och rapportera resultatet.
- Välj minst en relevant metric och motivera valet:
    - Regression: t.ex. MAE eller RMSE
    - Klassificering: t.ex. F1 eller recall/precision
- Sammanfatta resultat tydligt (tabell rekommenderas).
- Skriv en kort rekommendation: vilken modell skulle du ta vidare och varför?